# Aprendizado de regras associativas com Apriori

# 1. Configurações

Vamos fazer a importação de bibliotecas essenciais para análise de dados, visualização e modelagem:

- `numpy, pandas`: Para operações numéricas e manipulação e estruturação de dados.
- `seaborn, matplotlib`: Para criar gráficos e visualizações.
- `mlxtend`: Para modelar a mineração de regras associativas.

Essas importações preparam o ambiente para a mineração de regras.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

Este código usa o gdown para baixar um arquivo CSV do Google Drive:

- `URL`: O arquivo localizado na URL especificada será baixado.
- `-O ./<title>.csv`: O arquivo será salvo localmente com o nome \<title\>.csv.

Isso facilita o acesso direto aos dados sem a necessidade de downloads manuais.

In [ ]:
!gdown 1kE6RYPd14IPWL5GhhY05DihVhqtLPrxV -O ./movies.xls
!gdown 1u6XLcDqSBmm6T8bkdar53EeK-xfPSsnm -O ./ratings.csv

# 2. Leitura e exploração dos dados

Agora, vamos carregar os arquivo CSV baixados em DataFrames do pandas:

- `pd.read_csv`: Lê os arquivos .csv.

Em seguida, os DataFrames são exibidos para uma visualização inicial dos dados.

In [ ]:
df_movies = pd.read_csv("./movies.xls")
df_movies

In [ ]:
df_ratings = pd.read_csv("./ratings.csv")
df_ratings

# 3. Pré-processamento

Para facilitar a interpretação das análises, iniciamos fazendo uma troca dos IDs dos filmes do DataFrame de ratings pelos seus respectivos títulos. Para isso fazemos a operação de merge, combinando as duas tabelas sob a referência de um atributo em comum e listando apenas as colunas necessárias.

In [ ]:
df = df_ratings.merge(df_movies, on="movieId", how="left")
df = df[["userId", "title", "rating", "genres"]]
df

Em seguida, precisamos fazer uma análise das regras de negócio, por meio de uma filtragem dos filmes que os usuários deram notas altas, sob a hipótese de estes serem os seus filmes favoritos.

Dessa forma, o sistema de recomendação vai listar regras baseadas nas preferências quantitativas dos usuários.

In [ ]:
df_highratings = df[df["rating"] >= 4]
df_highratings

Em seguida, criamos extraímos dois arrays que contenham os filmes e usuários únicos registrados no sistema.

In [ ]:
movies = df["title"].unique()
movies

In [ ]:
users = df_ratings["userId"].unique()
users

Com estes dados em mãos, podemos criar um dicionário que armazene os dados das transações, que aqui são as avaliações que cada usuário deu para um conjunto de filmes, que entende-se que ele assistiu e gostou. Esse dicionário é então utilizado para montar um DataFrame de transações.

In [ ]:
# dicionário de transações
transactions_data = {movie_title: [] for movie_title in movies}

# para cada usuário
for user in users:
    # extrair um dataframe com as avaliações só deste usuário
    df_highratings_user = df_highratings[df_highratings["userId"] == user]

    # cria um conjunto com seus filmes favoritos
    favorite_movies = set(df_highratings_user["title"].values)

    # para cada filme do catálogo, verificar se é favorito para este usuário
    for movie_title in transactions_data.keys():
        if movie_title in favorite_movies:
            transactions_data[movie_title].append(True)
        else:
            transactions_data[movie_title].append(False)

df_transactions = pd.DataFrame(transactions_data)
df_transactions

# 4. Mineração de itemsets frequentes

Realizamos a mineração dos itemsets frequentes por meio do algoritmo Apriori, configurando um valor de suporte mínimo adequado. O resultado é um DataFrame com os k-itemsets cujo suporte é maior que o suporte mínimo fornecido.

In [ ]:
frequent_itemsets = apriori(df_transactions, min_support=0.1, use_colnames=True)
frequent_itemsets

# 5. Geração de regras associativas

Finalmente podemos gerar as regras, onde é interessante analisar estas da maior para a menor confiança.

In [ ]:
rules = association_rules(
    df=frequent_itemsets,
    # metric="confidence",
    # min_threshold=0.5
)
rules = rules.sort_values(by="confidence", ascending=False)
rules

Vamos salvar e baixar as regras para análise no Excel.

In [ ]:
rules.to_excel("regras.xlsx", index=False)

# 6. Regras por gênero de filme

Podemos também separar um subcojunto de transações que atendam a um determinado público-alvo ou segmento. Neste caso, vamos utilizar os diferentes gêneros de filmes para recomendar filmes de um mesmo gênero.

Primeiramente, vamos listar os gêneros registrados na base de dados.

In [ ]:
# unir todas as strings em uma única string
all_genres = "|".join(df_movies["genres"].values)

# dividir as strings e montar um conjunto de gêneros
genres = set(all_genres.split("|"))

genres

Para cada gênero, podemos criar colunas booleanas que indiquem se o filme trata de uma determinada temática.

In [ ]:
for genre in genres:
    df[genre] = df["genres"].str.contains(genre)
df

Agora escolhemos um gênero de interesse para seguir as análises. Um novo DataFrame é criado para conter apenas as review relativas àquele gênero de interesse. Esse DataFrame também passa por um filtro para conter apenas as reviews com nota de pelo menos 4 estrelas.

In [ ]:
genre_of_interest = "Comedy"

df_interest = df[df[genre_of_interest] == True]
df_interest = df_interest[df_interest["rating"] >= 4]
df_interest

Os títulos e usuários de interesse são extraídos novamente.

In [ ]:
movies_interest = df_interest["title"].unique()
users_interest = df_interest["userId"].unique()

E a mesmo DataFrame de transações é montado para o gênero de interesse.

In [ ]:
# dicionário de transações
transactions_interest_data = {movie_title: [] for movie_title in movies_interest}

# para cada usuário
for user in users_interest:
    # extrair um dataframe com as avaliações só deste usuário
    df_highratings_user = df_interest[df_interest["userId"] == user]

    # cria um conjunto com seus filmes favoritos
    favorite_movies = set(df_highratings_user["title"].values)

    # para cada filme do catálogo, verificar se é favorito para este usuário
    for movie_title in transactions_interest_data.keys():
        if movie_title in favorite_movies:
            transactions_interest_data[movie_title].append(True)
        else:
            transactions_interest_data[movie_title].append(False)

df_transactions_interest = pd.DataFrame(transactions_interest_data)
df_transactions_interest

Os itemsets frequentes são extraídos, de acordo com um suporte mínimo.

In [ ]:
frequent_itemsets_interest = apriori(df_transactions_interest, min_support=0.1, use_colnames=True)
frequent_itemsets_interest

E as regras são geradas a partir dos itemsets extraídos.

In [ ]:
rules_interest = association_rules(
    df=frequent_itemsets_interest,
    # metric="confidence",
    # min_threshold=0.5
)
rules_interest

Vamos salvar e baixar as regras para análise no Excel.

In [ ]:
rules_interest.to_excel("regras_interesse.xlsx", index=False)